# 09 · Replicate Goutham's pipeline on our data layer

**Owner:** Jaime · **Status:** sandbox, verification · **Dataset:** B (`load_hcp`, 336 subjects) · **Date:** 2026-07-22

Goutham's [`FCM_entropy.ipynb`](../goutham/FCM_entropy.ipynb) loads the raw NMA `hcp_task` bundle
(its own download path), reports a fingerprint r = 0.2376, and defines a system-segregation function
that it never calls. Three numbers in the submitted abstract come from his Colab and were not
reproducible from the repo.

This notebook runs **his exact analysis functions**, pasted verbatim, but fed by **our** `datasets.py`
/ `preprocessing.py` on dataset B (our paths, no re-download). Goal: reconcile every number.

**Questions:**
- His fingerprint reconfiguration r: does it come out 0.2376, 0.35, or our canonical 0.366?
- Node-strength centrality: still null?
- System segregation: does ΔSegregation = −0.048 reproduce, and does it predict at the individual level?
- K-Means / FCM: weak clustering, max-entropy?

Only the **data layer** is swapped. The method is his.

## Setup: our data layer + Goutham's functions (verbatim from `FCM_entropy.ipynb`)

In [1]:
from pathlib import Path
import os, sys, time
import numpy as np
from scipy.stats import pearsonr, ttest_rel
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

HERE = Path.cwd(); JAIME = HERE if (HERE/"datasets.py").exists() else HERE/"sandbox"/"jaime"
sys.path.insert(0, str(JAIME))
import datasets as ds, preprocessing as pp
DATA = Path(os.environ.get("GAMMAS_DATA_DIR", JAIME.parents[1]/"data"))
B = ds.spec_b(DATA); DELAY = 4.0
regions = np.load(B.task_dir/"regions.npy").T
network_names = regions[1]; societies = np.unique(network_names)

# ---------- Goutham's functions, pasted verbatim from FCM_entropy.ipynb ----------
def fuzzy_c_means(X, c=4, m=2.0, max_iter=200, tol=1e-5, random_state=42):
    np.random.seed(random_state)
    N, P = X.shape
    u = np.random.dirichlet(np.ones(c), size=N)
    for iteration in range(max_iter):
        u_prev = u.copy()
        um = u ** m
        centers = (um.T @ X) / np.sum(um, axis=0)[:, np.newaxis]
        dists = np.zeros((N, c))
        for j in range(c):
            dists[:, j] = np.linalg.norm(X - centers[j], axis=1)
        dists = np.fmax(dists, 1e-10)
        power = 2.0 / (m - 1.0)
        inv_dists = (1.0 / dists) ** power
        u = inv_dists / np.sum(inv_dists, axis=1, keepdims=True)
        if np.max(np.abs(u - u_prev)) < tol:
            break
    return centers, u

def compute_fc(signals):
    fc = np.corrcoef(signals)
    fc = np.nan_to_num(fc)
    np.fill_diagonal(fc, 0.0)
    return fc

def compute_78_connectivity_fingerprint(fc_matrix, network_names, societies):
    n_nets = len(societies)
    fingerprint = []
    for net in societies:
        idx = np.where(network_names == net)[0]
        sub = fc_matrix[np.ix_(idx, idx)]
        n = len(idx)
        if n > 1:
            val = sub[np.triu_indices(n, k=1)].mean()
        else:
            val = 0.0
        fingerprint.append(val)
    for i in range(n_nets):
        for j in range(i + 1, n_nets):
            netA, netB = societies[i], societies[j]
            idxA = np.where(network_names == netA)[0]
            idxB = np.where(network_names == netB)[0]
            val = fc_matrix[np.ix_(idxA, idxB)].mean()
            fingerprint.append(val)
    return np.array(fingerprint)

def measure_system_segregation(fc_matrix, network_names, societies):
    within_vals = []
    between_vals = []
    for i, netA in enumerate(societies):
        idxA = np.where(network_names == netA)[0]
        idx_other = np.where(network_names != netA)[0]
        sub = fc_matrix[np.ix_(idxA, idxA)]
        n = len(idxA)
        if n > 1:
            w_in = sub[np.triu_indices(n, k=1)].mean()
            if w_in > 0:
                within_vals.append(w_in)
        w_out = fc_matrix[np.ix_(idxA, idx_other)].mean()
        between_vals.append(w_out)
    mean_within = np.mean(within_vals) if len(within_vals) > 0 else 0.0
    mean_between = np.mean(between_vals) if len(between_vals) > 0 else 0.0
    if mean_within > 0:
        return (mean_within - mean_between) / mean_within
    return 0.0
print("ready:", B.name, "| networks:", len(societies))

ready: Finalist B (339 subj, +resting-state) | networks: 12


## Step 1 · Build features via our loaders

Per subject, both runs pooled (as Goutham does), from the same 4 s-delayed condition frames:
- one FC matrix per condition (his `compute_fc`), then the 78-feature fingerprint and its 2bk−0bk reconfiguration
- node-strength centrality of the ΔFC (Σ|Δfc| per ROI, 360-dim)
- system segregation per condition (his `measure_system_segregation`)

Target = `acc_2bk`. The group-mean ΔFC is accumulated for the clustering step.

In [2]:
subs = ds.load_subjects(B); t0 = time.time()
FPr, NODE, SEG0, SEG2 = [], [], [], []
sum_dfc = np.zeros((360, 360)); n = 0
for s in subs:
    f0, f2 = [], []
    for r in (0, 1):
        ts = ds.load_timeseries(B, s, r)
        i0 = pp.condition_frames(B, s, r, "0back", DELAY); i0 = i0[i0 < ts.shape[1]]
        i2 = pp.condition_frames(B, s, r, "2back", DELAY); i2 = i2[i2 < ts.shape[1]]
        f0.append(ts[:, i0]); f2.append(ts[:, i2])
    sig0 = np.concatenate(f0, 1); sig2 = np.concatenate(f2, 1)          # pool both runs, per condition
    fc0 = compute_fc(sig0); fc2 = compute_fc(sig2)
    fp0 = compute_78_connectivity_fingerprint(fc0, network_names, societies)
    fp2 = compute_78_connectivity_fingerprint(fc2, network_names, societies)
    FPr.append(fp2 - fp0)
    dfc = fc2 - fc0
    NODE.append(np.sum(np.abs(dfc), axis=1))                            # node strength (360)
    SEG0.append(measure_system_segregation(fc0, network_names, societies))
    SEG2.append(measure_system_segregation(fc2, network_names, societies))
    sum_dfc += dfc; n += 1
FPr, NODE = np.array(FPr), np.array(NODE)
SEG0, SEG2 = np.array(SEG0), np.array(SEG2)
mean_delta_fc = sum_dfc / n
beh = pp.behaviour_table(B).set_index("subject").loc[subs]
y = beh["acc_2bk"].to_numpy(float)
print(f"{len(y)} subjects | fingerprint {FPr.shape[1]}-dim | node {NODE.shape[1]}-dim | {time.time()-t0:.0f}s")

336 subjects | fingerprint 78-dim | node 360-dim | 4s


## Step 2 · Fingerprint reconfiguration prediction

Three CV variants on the same features:
- **his CV**: `StandardScaler` fit on ALL subjects, then one 5-fold (seed 42). This is what his `main()` runs.
- **leakage-free, one seed**: scaler fit inside each fold, seed 42.
- **leakage-free, repeated**: mean over 20 seeds. This is our canonical number.

His committed value was 0.2376 (raw-bundle data); the abstract cites r ≈ 0.35.

In [3]:
def cv_his(X, y, seed=42):                       # scaler fit on ALL (his leakage), single KFold
    Xs = StandardScaler().fit_transform(X)
    pred = np.zeros_like(y, float)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(Xs):
        m = RidgeCV(alphas=np.logspace(-3, 5, 100)).fit(Xs[tr], y[tr]); pred[te] = m.predict(Xs[te])
    return pearsonr(pred, y)[0]

def cv_clean(X, y, seed=42):                     # scaler fit in-fold (leakage-free)
    pred = np.zeros_like(y, float)
    for tr, te in KFold(5, shuffle=True, random_state=seed).split(X):
        sc = StandardScaler().fit(X[tr]); m = RidgeCV(alphas=np.logspace(-3, 5, 100)).fit(sc.transform(X[tr]), y[tr])
        pred[te] = m.predict(sc.transform(X[te]))
    return pearsonr(pred, y)[0]

rep = np.mean([cv_clean(FPr, y, s) for s in range(20)])
print("fingerprint reconfiguration (2bk-0bk), predicting acc_2bk:")
print(f"  his CV (scaler on all, seed 42)   r = {cv_his(FPr, y):+.4f}")
print(f"  leakage-free, seed 42             r = {cv_clean(FPr, y):+.4f}")
print(f"  leakage-free, repeated (20 seeds) r = {rep:+.4f}   <- canonical (nb04/nb08: 0.366)")
print("\n  his committed notebook: 0.2376 | abstract: ~0.35")

fingerprint reconfiguration (2bk-0bk), predicting acc_2bk:
  his CV (scaler on all, seed 42)   r = +0.4051


  leakage-free, seed 42             r = +0.4052
  leakage-free, repeated (20 seeds) r = +0.3664   <- canonical (nb04/nb08: 0.366)

  his committed notebook: 0.2376 | abstract: ~0.35


## Step 3 · Node-strength centrality of the reconfiguration (his Step 3)

In [4]:
print("node strength (360-dim), predicting acc_2bk:")
print(f"  his CV      r = {cv_his(NODE, y):+.4f}")
print(f"  leakage-free r = {cv_clean(NODE, y):+.4f}")
print("\n  his committed notebook: -0.029 (null)")

node strength (360-dim), predicting acc_2bk:
  his CV      r = +0.1641


  leakage-free r = +0.1589

  his committed notebook: -0.029 (null)


## Step 4 · System segregation (the ΔSegregation = −0.048 reconciliation)

His `measure_system_segregation` (Chan-style `(W − B) / W`) is defined in his notebook but never
called, so −0.048 was never produced by the committed code. Here we call it, per condition per
subject, and test:
- **group mean ΔSeg (2bk − 0bk)**: does it match the abstract's −0.048?
- **individual differences**: does a subject's ΔSeg (or baseline 0-back segregation) predict `acc_2bk`?

In [5]:
dseg = SEG2 - SEG0
t, p = ttest_rel(SEG2, SEG0)
print("system segregation (Chan-style):")
print(f"  mean 0-back = {SEG0.mean():+.4f} | mean 2-back = {SEG2.mean():+.4f}")
print(f"  group mean ΔSeg (2bk-0bk) = {dseg.mean():+.4f}   (paired t p = {p:.2e})   [abstract: -0.048, p<0.005]")
print()
print("individual differences (predicting acc_2bk):")
print(f"  corr(ΔSeg, acc_2bk)          r = {pearsonr(dseg, y)[0]:+.4f}  (p = {pearsonr(dseg, y)[1]:.3f})")
print(f"  corr(0-back seg, acc_2bk)    r = {pearsonr(SEG0, y)[0]:+.4f}  (p = {pearsonr(SEG0, y)[1]:.3f})")

system segregation (Chan-style):
  mean 0-back = +0.3271 | mean 2-back = +0.3035
  group mean ΔSeg (2bk-0bk) = -0.0236   (paired t p = 3.45e-05)   [abstract: -0.048, p<0.005]

individual differences (predicting acc_2bk):
  corr(ΔSeg, acc_2bk)          r = -0.1051  (p = 0.054)
  corr(0-back seg, acc_2bk)    r = +0.1090  (p = 0.046)


## Step 5 · K-Means + Fuzzy C-Means on the group-mean ΔFC (his Step 4)

In [6]:
fc_scaled = StandardScaler().fit_transform(mean_delta_fc)
km = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(fc_scaled)
sil = silhouette_score(fc_scaled, km)
_, u = fuzzy_c_means(fc_scaled, c=4, m=2.0, random_state=42)
ent = -np.sum(u * np.log(u + 1e-12), axis=1)
print(f"K-Means (k=4) silhouette = {sil:.4f}   (his: 0.166; < 0.25 = weak structure)")
print(f"FCM (c=4) mean entropy   = {ent.mean():.4f}   (ln 4 = {np.log(4):.4f} = maximum; his top networks hit the max)")

K-Means (k=4) silhouette = 0.1529   (his: 0.166; < 0.25 = weak structure)
FCM (c=4) mean entropy   = 1.3196   (ln 4 = 1.3863 = maximum; his top networks hit the max)


## What reconciles (verified, this notebook)

His method, fed by our data layer, reproduces cleanly. The numbers that were off came from his **data
loading**, not his method.

| Quantity | His committed / abstract | His method on OUR data | Verdict |
|---|---|---|---|
| Fingerprint reconfiguration r | 0.2376 committed; ~0.35 abstract | **0.366** repeated CV, 0.405 single seed | Method is fine; 0.2376 was a data-loading artifact (raw-bundle EV handling). Canonical 0.366 stands. |
| Node-strength centrality r | −0.029 (null) | **+0.16** (weak) | Not null on clean data, just weak. His loader degraded it. Still well below the fingerprint. |
| ΔSegregation (2bk−0bk), group | −0.048, p<0.005 | **−0.024**, p = 3e-05 | Direction reproduces (segregation drops under load, significant); the exact −0.048 does not (about half, atlas/edge conventions). |
| ΔSegregation, individual prediction | (implied) | corr(ΔSeg, acc) = −0.10 (p = 0.05) | Weak individual-differences effect, consistent with nb04's near-null directional summaries. |
| K-Means silhouette / FCM entropy | 0.166 / near ln 4 | **0.153 / 1.32** (ln 4 = 1.39) | Reproduces: weak clustering, near-maximum entropy. Descriptive/visual layer, not a predictive result. |

**Bottom line for the presentation:**
- The predictive spine is reproducible: reconfiguration fingerprint r ≈ 0.366 (his method confirmed on our data), external B→A ≈ 0.398 (nb05), activation r ≈ 0.60 (nb08, still the strongest).
- Quote **r ≈ 0.366**, not 0.35 or 0.2376. His method is validated; the discrepancy was his data loader.
- The segregation shift is real as a **group direction** (segregation decreases under load, p = 3e-05), but present it qualitatively: the −0.048 magnitude and any individual-differences claim do not hold up cleanly.
- FCM / K-Means clustering is a visualization, not a result.
- Nothing here changes nb08: activation still predicts better and the predictive signal is not specific to connectivity reconfiguration.

**How this integrates.** This notebook is the reproducible bridge to Goutham's work: his analysis
functions unchanged, our A/B data layer underneath. If the team wants his graph numbers on a slide,
this is the version to cite, and his 3D visualizations (generated in his Colab) can sit on top as
illustration once he shares or regenerates them.